## Implementation of Deep Learning Model


In [ ]:
# My informations here!
student_name = 'Nirodha Samdaruwani'
student_email = "ad9906@student.jamk.fi"

In [97]:
import pandas as pd
import random

input_file = "data/AI_human_cleaned.csv"
output_file = "data/AI_human_subset_10k.csv"

subset_size = 10000
chunk_size = 5000   # read 5k rows at a time

subset = []

reader = pd.read_csv(input_file, chunksize=chunk_size)

for chunk in reader:
    # Randomly sample from each chunk
    sample = chunk.sample(
        n=min(len(chunk), int(subset_size / 50)),  # take small pieces from each chunk
        random_state=42
    )
    subset.append(sample)

# Combine collected samples
df_subset = pd.concat(subset, ignore_index=True)

# Finally sample down to EXACT 10k rows
df_subset = df_subset.sample(n=subset_size, random_state=42)

# Save result
df_subset.to_csv(output_file, index=False)

print("Subset saved:", output_file)
print("Subset shape:", df_subset.shape)


Subset saved: data/AI_human_subset_10k.csv
Subset shape: (10000, 4)


Implement and compare multiple deep learning models for classifying
texts as **AI-generated (1)** or **human-written (0)** using a cleaned subset of 10,000 samples.

We will:
- Load the preprocessed subset (`AI_human_subset_10k.csv`)
- Split data into train and test sets
- Tokenize and pad the text
- Train and evaluate 3 models:
  1. BiLSTM v2 (enhanced + bidirectional)
  2. 1D CNN
  3. GRU v3 (stacked bidirectional GRU)
  4. FastText-like shallow network
  5. Hybrid CNN+GRU model
- Calculate accuracy and confusion matrices
- Collect results for comparison (as suggested in assignment instructions)

#### 1. Import Libraries & Load Subset Dataset

In [98]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
import time

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

pd.set_option('display.max_colwidth', 200)

# Load the 10k subset created earlier
df = pd.read_csv("data/AI_human_subset_10k.csv")

print("Subset shape:", df.shape)
print(df.columns)
df.head()

Subset shape: (10000, 4)
Index(['text', 'generated', 'clean_text', 'word_count'], dtype='object')


,text,generated,clean_text,word_count
0,"Limiting car usage is an important stop towards reducing our carbon footprint and making our cities more livable. Cars ARX a major source of air pollution and greenhouse gas Missions, which contr...",1.0,limiting car usage is an important stop towards reducing our carbon footprint and making our cities more livable cars arx a major source of air pollution and greenhouse gas missions which contribu...,209
1,"Genus is one of the closest planet to Earth, second in comparison to Mars who switches that role every once in a while. Genus is considered to be one of the most Earthlike planets that is in our s...",0.0,genus is one of the closest planet to earth second in comparison to mars who switches that role every once in a while genus is considered to be one of the most earthlike planets that is in our sol...,675
2,"Hey, Mrs. Johnson! Here's my essay of how a person's past or environment CAU affect their first impression AUD how it may be possible to change it.\n\nFirst impressions are super important, right?...",1.0,hey mrs johnson heres my essay of how a persons past or environment cau affect their first impression aud how it may be possible to change it first impressions are super important right like they ...,360
3,"Near the French and Swiss borders a new district on the outskirts of Freiburg street parking, driveways, and home barges are forbidden. Y person that would like a car would have to pay 40,000 for ...",0.0,near the french and swiss borders a new district on the outskirts of freiburg street parking driveways and home barges are forbidden y person that would like a car would have to pay for a parking ...,257
4,"The university education is not only for preparing students for employment. It is also for teaching them how to think critically and solve problems. For example, a student who attends a university...",1.0,the university education is not only for preparing students for employment it is also for teaching them how to think critically and solve problems for example a student who attends a university ma...,218


#### 2. Set Global Random Seed for Reproducibility

In [100]:
import random
import numpy as np
import tensorflow as tf

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

#### 3. Prepare Texts and Labels

We will use:
- `clean_text` as input feature
- `generated` as the target label (0 = Human, 1 = AI)


In [101]:
# Use cleaned text and labels
texts = df['clean_text'].astype(str).tolist()
labels = df['generated'].astype(int).values

print("Number of samples:", len(texts))
print("Label distribution:")
print(pd.Series(labels).value_counts())


Number of samples: 10000
Label distribution:
0    6325
1    3675
Name: count, dtype: int64


#### 4. Train-Test Split

We split the dataset into:
- 80% training data
- 20% test data

We use stratified splitting to preserve class balance.


In [102]:
X_train, X_test, y_train, y_test = train_test_split(
    texts, labels,
    test_size=0.20,
    random_state=42,
    stratify=labels
)

len(X_train), len(X_test)


(8000, 2000)

#### 5. Tokenization and Sequence Padding

We convert the text into integer sequences using Keras `Tokenizer`, then pad them
to a fixed maximum length.

Configuration:
- Vocabulary size: 20,000 words
- Maximum sequence length: 256 tokens


In [103]:
MAX_WORDS = 20000
MAX_LEN = 256

# Tokenizer on training data
tokenizer = Tokenizer(num_words=MAX_WORDS, oov_token="<OOV>")
tokenizer.fit_on_texts(X_train)

print("Vocabulary size (tokenizer.word_index):", len(tokenizer.word_index))

# Convert to sequences
train_seq = tokenizer.texts_to_sequences(X_train)
test_seq = tokenizer.texts_to_sequences(X_test)

# Pad sequences
train_pad = pad_sequences(train_seq, maxlen=MAX_LEN, padding='post', truncating='post')
test_pad = pad_sequences(test_seq, maxlen=MAX_LEN, padding='post', truncating='post')

train_pad.shape, test_pad.shape


Vocabulary size (tokenizer.word_index): 34328


((8000, 256), (2000, 256))

#### 6. Model Configuration Hyperparameters

To keep training efficient on a normal laptop, I use:

- `EMBEDDING_DIM = 64`
- `BATCH_SIZE = 32`
- `EPOCHS = 3` (base; some models use up to 6 with EarlyStopping)


In [ ]:
EMBEDDING_DIM = 64     # stronger embedding for all models
BATCH_SIZE = 32
EPOCHS = 3             # set custom epochs for v2 LSTM later if needed

results = []  # to store model name, accuracy, training time

#### 7. Model 1 - BiLSTM Architecture + Training

Architecture:
- Embedding layer
- BiLSTM(64)
- Dense layers + dropout regularization
- Adam optimizer (LR = 2e-4)
- EarlyStopping to prevent overfitting

This model captures sequential patterns in the text.


In [ ]:
from tensorflow.keras import layers, optimizers, callbacks

def build_lstm_model():
    model = tf.keras.Sequential([
        layers.Embedding(MAX_WORDS, EMBEDDING_DIM),
        layers.Bidirectional(layers.LSTM(64)),
        layers.Dropout(0.4),
        layers.Dense(64, activation='relu'),
        layers.Dropout(0.3),
        layers.Dense(1, activation='sigmoid')
    ])

    opt = optimizers.Adam(learning_rate=2e-4)

    model.compile(
        loss='binary_crossentropy',
        optimizer=opt,
        metrics=['accuracy']
    )

    return model


# Build model
model_lstm = build_lstm_model()

model_lstm.build(input_shape=(None, MAX_LEN))

model_lstm.summary()

# Early stopping
early_stop = callbacks.EarlyStopping(
    monitor='val_loss',
    patience=2,
    restore_best_weights=True
)

# Train model
start_time = time.time()
history_lstm = model_lstm.fit(
    train_pad, y_train,
    epochs=6,
    batch_size=BATCH_SIZE,
    validation_split=0.1,
    callbacks=[early_stop],
    verbose=1
)

train_time_lstm = time.time() - start_time
print(f"\nLSTM training time: {train_time_lstm:.2f} seconds")


Model: "sequential_31"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_31 (Embedding)        │ (None, 256, 64)        │     1,280,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional_13                │ (None, 128)            │        66,048 │
│ (Bidirectional)                 │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_41 (Dropout)            │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_60 (Dense)                │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_42 (Dropout)            │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_61 (Dense)                │ (None, 1)              │            65 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,354,369 (5.17 MB)

 Trainable params: 1,354,369 (5.17 MB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/6
225/225 ━━━━━━━━━━━━━━━━━━━━ 86s 311ms/step - accuracy: 0.7328 - loss: 0.5339 - val_accuracy: 0.8938 - val_loss: 0.2756
Epoch 2/6
225/225 ━━━━━━━━━━━━━━━━━━━━ 69s 308ms/step - accuracy: 0.9507 - loss: 0.1657 - val_accuracy: 0.9500 - val_loss: 0.1556
Epoch 3/6
225/225 ━━━━━━━━━━━━━━━━━━━━ 63s 280ms/step - accuracy: 0.9347 - loss: 0.2143 - val_accuracy: 0.9588 - val_loss: 0.1179
Epoch 4/6
225/225 ━━━━━━━━━━━━━━━━━━━━ 68s 303ms/step - accuracy: 0.9768 - loss: 0.0971 - val_accuracy: 0.9675 - val_loss: 0.1014
Epoch 5/6
225/225 ━━━━━━━━━━━━━━━━━━━━ 63s 280ms/step - accuracy: 0.9833 - loss: 0.0628 - val_accuracy: 0.9613 - val_loss: 0.1120
Epoch 6/6
225/225 ━━━━━━━━━━━━━━━━━━━━ 68s 303ms/step - accuracy: 0.9839 - loss: 0.0577 - val_accuracy: 0.9700 - val_loss: 0.1016

LSTM training time: 417.96 seconds


##### 7.1 BiLSTM Evaluation: Accuracy & Confusion Matrix

In [123]:
# Predictions (probabilities -> class labels)
y_pred_lstm_prob = model_lstm.predict(test_pad)
y_pred_lstm = (y_pred_lstm_prob > 0.5).astype("int32")

acc_lstm = accuracy_score(y_test, y_pred_lstm)
cm_lstm = confusion_matrix(y_test, y_pred_lstm)

print("LSTM Accuracy:", acc_lstm)
print("\nLSTM Confusion Matrix:\n", cm_lstm)

# Store results
results.append({
    "Model": "LSTM",
    "Accuracy": acc_lstm,
    "Train Time (s)": train_time_lstm
})


63/63 ━━━━━━━━━━━━━━━━━━━━ 6s 88ms/step
LSTM Accuracy: 0.9725

LSTM Confusion Matrix:
 [[1228   37]
 [  18  717]]


#### 8. Model 2 - 1D CNN Architecture + Training

Architecture:
- Embedding
- Conv1D(64, kernel=5)
- GlobalMaxPooling1D
- Dense classifier head

This model captures local n-gram patterns (like phrases).


In [ ]:
def build_cnn_model():
    model = tf.keras.Sequential([
        tf.keras.layers.Embedding(MAX_WORDS, EMBEDDING_DIM),
        tf.keras.layers.Conv1D(filters=64, kernel_size=5, activation='relu'),
        tf.keras.layers.GlobalMaxPooling1D(),
        tf.keras.layers.Dense(64, activation='relu'),
        tf.keras.layers.Dropout(0.3),
        tf.keras.layers.Dense(1, activation='sigmoid')
    ])

    model.compile(
        loss='binary_crossentropy',
        optimizer='adam',
        metrics=['accuracy']
    )

    return model


# Build model
model_cnn = build_cnn_model()

model_cnn.build(input_shape=(None, MAX_LEN))

model_cnn.summary()

# Train
start_time = time.time()
history_cnn = model_cnn.fit(
    train_pad, y_train,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    validation_split=0.1,
    verbose=1
)

train_time_cnn = time.time() - start_time
print(f"\nCNN training time: {train_time_cnn:.2f} seconds")


Model: "sequential_32"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_32 (Embedding)        │ (None, 256, 64)        │     1,280,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_10 (Conv1D)              │ (None, 252, 64)        │        20,544 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_max_pooling1d_6          │ (None, 64)             │             0 │
│ (GlobalMaxPooling1D)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_62 (Dense)                │ (None, 64)             │         4,160 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_43 (Dropout)            │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_63 (Dense)                │ (None, 1)              │            65 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,304,769 (4.98 MB)

 Trainable params: 1,304,769 (4.98 MB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/3
225/225 ━━━━━━━━━━━━━━━━━━━━ 13s 46ms/step - accuracy: 0.7918 - loss: 0.4162 - val_accuracy: 0.9575 - val_loss: 0.1358
Epoch 2/3
225/225 ━━━━━━━━━━━━━━━━━━━━ 10s 44ms/step - accuracy: 0.9708 - loss: 0.0868 - val_accuracy: 0.9725 - val_loss: 0.0742
Epoch 3/3
225/225 ━━━━━━━━━━━━━━━━━━━━ 11s 47ms/step - accuracy: 0.9949 - loss: 0.0210 - val_accuracy: 0.9725 - val_loss: 0.0729

CNN training time: 33.50 seconds


#### 8.1 CNN Evaluation: Accuracy & Confusion Matrix


In [125]:
y_pred_cnn_prob = model_cnn.predict(test_pad)
y_pred_cnn = (y_pred_cnn_prob > 0.5).astype("int32")

acc_cnn = accuracy_score(y_test, y_pred_cnn)
cm_cnn = confusion_matrix(y_test, y_pred_cnn)

print("CNN Accuracy:", acc_cnn)
print("\nCNN Confusion Matrix:\n", cm_cnn)

results.append({
    "Model": "CNN",
    "Accuracy": acc_cnn,
    "Train Time (s)": train_time_cnn
})


63/63 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step
CNN Accuracy: 0.9775

CNN Confusion Matrix:
 [[1244   21]
 [  24  711]]


#### 9. Model 3 - GRU (V3) Architecture + Training

Architecture:
- BiGRU(64) → BiGRU(32)
- Multiple dropout layers
- Dense layer with 128 units
- Adam optimizer with reduced learning rate

GRU is a variant of RNN similar to LSTM, often lighter and faster.


In [ ]:
def build_gru_model_v3():
    model = tf.keras.Sequential([
        tf.keras.layers.Embedding(MAX_WORDS, EMBEDDING_DIM),

        # Stronger GRU encoder
        tf.keras.layers.Bidirectional(tf.keras.layers.GRU(
            64, return_sequences=True, dropout=0.2, recurrent_dropout=0.2)),
        
        tf.keras.layers.Bidirectional(tf.keras.layers.GRU(
            32, return_sequences=False, dropout=0.2, recurrent_dropout=0.2)),

        # Dense head
        tf.keras.layers.Dense(128, activation='relu'),
        tf.keras.layers.Dropout(0.4),

        tf.keras.layers.Dense(1, activation='sigmoid')
    ])

    model.compile(
        loss='binary_crossentropy',
        optimizer=tf.keras.optimizers.Adam(learning_rate=2e-4),
        metrics=['accuracy']
    )

    return model


# Build model
model_gru = build_gru_model_v3()

model_gru.build(input_shape=(None, MAX_LEN))

model_gru.summary()


# Early stopping
early_stop_gru = tf.keras.callbacks.EarlyStopping(
    monitor='val_loss',
    patience=2,
    restore_best_weights=True
)

# Train model
start_time = time.time()
history_gru = model_gru.fit(
    train_pad, y_train,
    epochs=6,
    batch_size=BATCH_SIZE,
    validation_split=0.1,
    callbacks=[early_stop_gru],
    verbose=1
)

train_time_gru = time.time() - start_time
print(f"\nGRU training time: {train_time_gru:.2f} seconds")


Model: "sequential_33"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_33 (Embedding)        │ (None, 256, 64)        │     1,280,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional_14                │ (None, 256, 128)       │        49,920 │
│ (Bidirectional)                 │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional_15                │ (None, 64)             │        31,104 │
│ (Bidirectional)                 │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_64 (Dense)                │ (None, 128)            │         8,320 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_44 (Dropout)            │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_65 (Dense)                │ (None, 1)              │           129 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,369,473 (5.22 MB)

 Trainable params: 1,369,473 (5.22 MB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/6
225/225 ━━━━━━━━━━━━━━━━━━━━ 212s 851ms/step - accuracy: 0.6889 - loss: 0.5824 - val_accuracy: 0.8275 - val_loss: 0.3613
Epoch 2/6
225/225 ━━━━━━━━━━━━━━━━━━━━ 177s 783ms/step - accuracy: 0.9157 - loss: 0.2261 - val_accuracy: 0.9150 - val_loss: 0.2220
Epoch 3/6
225/225 ━━━━━━━━━━━━━━━━━━━━ 169s 751ms/step - accuracy: 0.9600 - loss: 0.1159 - val_accuracy: 0.9275 - val_loss: 0.2031
Epoch 4/6
225/225 ━━━━━━━━━━━━━━━━━━━━ 190s 845ms/step - accuracy: 0.9765 - loss: 0.0723 - val_accuracy: 0.9287 - val_loss: 0.2022
Epoch 5/6
225/225 ━━━━━━━━━━━━━━━━━━━━ 172s 762ms/step - accuracy: 0.9860 - loss: 0.0460 - val_accuracy: 0.9275 - val_loss: 0.2090
Epoch 6/6
225/225 ━━━━━━━━━━━━━━━━━━━━ 195s 867ms/step - accuracy: 0.9890 - loss: 0.0375 - val_accuracy: 0.9362 - val_loss: 0.2143

GRU training time: 1114.68 seconds


#### 9.1 GRU Evaluation: Accuracy & Confusion Matrix


In [127]:
y_pred_gru_prob = model_gru.predict(test_pad)
y_pred_gru = (y_pred_gru_prob > 0.5).astype("int32")

acc_gru = accuracy_score(y_test, y_pred_gru)
cm_gru = confusion_matrix(y_test, y_pred_gru)

print("GRU Accuracy:", acc_gru)
print("\nGRU Confusion Matrix:\n", cm_gru)

results.append({
    "Model": "GRU",
    "Accuracy": acc_gru,
    "Train Time (s)": train_time_gru
})


63/63 ━━━━━━━━━━━━━━━━━━━━ 15s 191ms/step
GRU Accuracy: 0.952

GRU Confusion Matrix:
 [[1220   45]
 [  51  684]]


#### 10. Model 4 - FastText-Like Shallow Network + Training

- Embedding
- GlobalAveragePooling1D
- Dense classification

In [ ]:
def build_fasttext_model():
    model = tf.keras.Sequential([
        tf.keras.layers.Embedding(MAX_WORDS, EMBEDDING_DIM),
        tf.keras.layers.GlobalAveragePooling1D(),
        tf.keras.layers.Dense(64, activation='relu'),
        tf.keras.layers.Dropout(0.3),
        tf.keras.layers.Dense(1, activation='sigmoid')
    ])

    model.compile(
        loss='binary_crossentropy',
        optimizer='adam',
        metrics=['accuracy']
    )
    
    return model


# Build model
model_fast = build_fasttext_model()

model_fast.build(input_shape=(None, MAX_LEN))

model_fast.summary()


# Train the model 
start_time = time.time()
history_fast = model_fast.fit(
    train_pad, y_train,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    validation_split=0.1,
    verbose=1
)

train_time_fast = time.time() - start_time
print(f"\nFastText-like model training time: {train_time_fast:.2f} seconds")


Model: "sequential_34"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_34 (Embedding)        │ (None, 256, 64)        │     1,280,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling1d_6      │ (None, 64)             │             0 │
│ (GlobalAveragePooling1D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_66 (Dense)                │ (None, 64)             │         4,160 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_45 (Dropout)            │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_67 (Dense)                │ (None, 1)              │            65 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,284,225 (4.90 MB)

 Trainable params: 1,284,225 (4.90 MB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/3
225/225 ━━━━━━━━━━━━━━━━━━━━ 10s 33ms/step - accuracy: 0.8367 - loss: 0.3605 - val_accuracy: 0.9638 - val_loss: 0.1156
Epoch 2/3
225/225 ━━━━━━━━━━━━━━━━━━━━ 7s 30ms/step - accuracy: 0.9750 - loss: 0.0841 - val_accuracy: 0.9775 - val_loss: 0.0689
Epoch 3/3
225/225 ━━━━━━━━━━━━━━━━━━━━ 7s 29ms/step - accuracy: 0.9872 - loss: 0.0435 - val_accuracy: 0.9825 - val_loss: 0.0549

FastText-like model training time: 23.46 seconds


#### 10.1 FastText-Like Evaluation & Metrics

In [129]:
y_pred_fast_prob = model_fast.predict(test_pad)
y_pred_fast = (y_pred_fast_prob > 0.5).astype("int32")

acc_fast = accuracy_score(y_test, y_pred_fast)
cm_fast = confusion_matrix(y_test, y_pred_fast)

print("FastText-like Model Accuracy:", acc_fast)
print("\nFastText-like Model Confusion Matrix:\n", cm_fast)

results.append({
    "Model": "FastText-like",
    "Accuracy": acc_fast,
    "Train Time (s)": train_time_fast
})


63/63 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step
FastText-like Model Accuracy: 0.9825

FastText-like Model Confusion Matrix:
 [[1242   23]
 [  12  723]]


#### 11. Model 5 - Hybrid CNN + GRU Architecture + Training

- CNN block (extracts local phrase-level features)
- GRU block (captures sequential dependencies)
- Dense layers with dropout

In [ ]:
def build_cnn_gru_model():
    model = tf.keras.Sequential([
        # Shared embedding layer
        tf.keras.layers.Embedding(MAX_WORDS, EMBEDDING_DIM),

        # CNN part – local feature extractor
        tf.keras.layers.Conv1D(filters=64, kernel_size=5, activation='relu'),
        tf.keras.layers.MaxPooling1D(pool_size=2),

        # GRU part – sequence modeling
        tf.keras.layers.GRU(64, dropout=0.2, recurrent_dropout=0.2),

        # Dense head
        tf.keras.layers.Dense(64, activation='relu'),
        tf.keras.layers.Dropout(0.3),

        tf.keras.layers.Dense(1, activation='sigmoid')
    ])

    model.compile(
        loss='binary_crossentropy',
        optimizer=tf.keras.optimizers.Adam(learning_rate=2e-4),
        metrics=['accuracy']
    )
    
    return model


# Build model
model_hybrid = build_cnn_gru_model()

model_hybrid.build(input_shape=(None, MAX_LEN))

model_hybrid.summary()


# Early stopping for the hybrid model
early_stop_hybrid = tf.keras.callbacks.EarlyStopping(
    monitor='val_loss',
    patience=2,
    restore_best_weights=True
)

# Train model
start_time = time.time()
history_hybrid = model_hybrid.fit(
    train_pad, y_train,
    epochs=6,
    batch_size=BATCH_SIZE,
    validation_split=0.1,
    callbacks=[early_stop_hybrid],
    verbose=1
)

train_time_hybrid = time.time() - start_time
print(f"\nCNN+GRU Hybrid training time: {train_time_hybrid:.2f} seconds")


Model: "sequential_35"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_35 (Embedding)        │ (None, 256, 64)        │     1,280,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_11 (Conv1D)              │ (None, 252, 64)        │        20,544 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d_4 (MaxPooling1D)  │ (None, 126, 64)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ gru_17 (GRU)                    │ (None, 64)             │        24,960 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_68 (Dense)                │ (None, 64)             │         4,160 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_46 (Dropout)            │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_69 (Dense)                │ (None, 1)              │            65 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,329,729 (5.07 MB)

 Trainable params: 1,329,729 (5.07 MB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/6
225/225 ━━━━━━━━━━━━━━━━━━━━ 61s 148ms/step - accuracy: 0.6289 - loss: 0.6548 - val_accuracy: 0.6425 - val_loss: 0.6073
Epoch 2/6
225/225 ━━━━━━━━━━━━━━━━━━━━ 29s 129ms/step - accuracy: 0.8271 - loss: 0.3855 - val_accuracy: 0.9513 - val_loss: 0.1250
Epoch 3/6
225/225 ━━━━━━━━━━━━━━━━━━━━ 28s 122ms/step - accuracy: 0.9431 - loss: 0.1722 - val_accuracy: 0.9638 - val_loss: 0.1132
Epoch 4/6
225/225 ━━━━━━━━━━━━━━━━━━━━ 28s 124ms/step - accuracy: 0.9693 - loss: 0.1088 - val_accuracy: 0.9613 - val_loss: 0.1148
Epoch 5/6
225/225 ━━━━━━━━━━━━━━━━━━━━ 27s 118ms/step - accuracy: 0.9701 - loss: 0.1018 - val_accuracy: 0.9663 - val_loss: 0.1090
Epoch 6/6
225/225 ━━━━━━━━━━━━━━━━━━━━ 31s 138ms/step - accuracy: 0.9789 - loss: 0.0754 - val_accuracy: 0.9600 - val_loss: 0.1101

CNN+GRU Hybrid training time: 203.53 seconds


#### 11.1 CNN+GRU Hybrid Evaluation & Metrics

In [131]:
y_pred_hybrid_prob = model_hybrid.predict(test_pad)
y_pred_hybrid = (y_pred_hybrid_prob > 0.5).astype("int32")

acc_hybrid = accuracy_score(y_test, y_pred_hybrid)
cm_hybrid = confusion_matrix(y_test, y_pred_hybrid)

print("CNN+GRU Hybrid Accuracy:", acc_hybrid)
print("\nCNN+GRU Hybrid Confusion Matrix:\n", cm_hybrid)

results.append({
    "Model": "CNN+GRU Hybrid",
    "Accuracy": acc_hybrid,
    "Train Time (s)": train_time_hybrid
})


63/63 ━━━━━━━━━━━━━━━━━━━━ 4s 50ms/step
CNN+GRU Hybrid Accuracy: 0.9765

CNN+GRU Hybrid Confusion Matrix:
 [[1250   15]
 [  32  703]]


#### 12. Compare Model Performance

We compare:
- Accuracy on test set
- Training times

This satisfies the requirement to "keep a notebook of different runs and configurations".


In [137]:
results = [
    {"Model": "LSTM", "Accuracy": acc_lstm, "Train Time (s)": train_time_lstm},
    {"Model": "CNN", "Accuracy": acc_cnn, "Train Time (s)": train_time_cnn},
    {"Model": "GRU", "Accuracy": acc_gru, "Train Time (s)": train_time_gru},
    {"Model": "FastText-like", "Accuracy": acc_fast, "Train Time (s)": train_time_fast},
    {"Model": "CNN+GRU Hybrid", "Accuracy": acc_hybrid, "Train Time (s)": train_time_hybrid}
]

results_df = pd.DataFrame(results)
results_df


,Model,Accuracy,Train Time (s)
0,LSTM,0.9725,417.959147
1,CNN,0.9775,33.503355
2,GRU,0.9520,1114.678786
3,FastText-like,0.9825,23.460270
4,CNN+GRU Hybrid,0.9765,203.527962


In [138]:
results_df.sort_values(by="Accuracy", ascending=False)

,Model,Accuracy,Train Time (s)
3,FastText-like,0.9825,23.460270
1,CNN,0.9775,33.503355
4,CNN+GRU Hybrid,0.9765,203.527962
0,LSTM,0.9725,417.959147
2,GRU,0.9520,1114.678786


In [139]:
results_df = pd.DataFrame(results)
results_df.sort_values(by="Accuracy", ascending=False).reset_index(drop=True)


,Model,Accuracy,Train Time (s)
0,FastText-like,0.9825,23.460270
1,CNN,0.9775,33.503355
2,CNN+GRU Hybrid,0.9765,203.527962
3,LSTM,0.9725,417.959147
4,GRU,0.9520,1114.678786


In [140]:
import pickle

artifacts = {
    "results": results,                         # list of dicts with Model, Accuracy, Train Time
    "history_lstm": history_lstm.history,
    "history_cnn": history_cnn.history,
    "history_gru": history_gru.history,
    "history_fast": history_fast.history,
    "history_hybrid": history_hybrid.history,
    "cm_lstm": cm_lstm,
    "cm_cnn": cm_cnn,
    "cm_gru": cm_gru,
    "cm_fast": cm_fast,
    "cm_hybrid": cm_hybrid,
}

with open("data/dl_artifacts.pkl", "wb") as f:
    pickle.dump(artifacts, f)

print("Saved deep learning artifacts to data/dl_artifacts.pkl")


Saved deep learning artifacts to data/dl_artifacts.pkl
